# Unsupervised Learning — How to Evaluate Clustering Without Labels?
## Iris example: internal criteria, stability, and external labels

Notebook này minh họa đúng ý trong slide:

- **Internal criteria**: inertia, silhouette score.
- **Stability**: cluster có còn giống khi perturb/resample dữ liệu không?
- **External labels**: nếu có nhãn, chỉ dùng **sau khi clustering xong** để đánh giá.

> Quan trọng: species labels của Iris **không được dùng để fit KMeans**.

## 1. Ý tưởng chính

Trong supervised learning, ta có thể so sánh dự đoán với nhãn thật.

Trong clustering, thường không có nhãn thật, nên không thể đánh giá bằng accuracy như classification.

Ta có 3 hướng:

### A. Internal criteria
Chỉ dùng dữ liệu và kết quả clustering.

### B. Stability
Perturb/resample dữ liệu rồi clustering lại xem kết quả có ổn định không.

### C. External labels
Nếu có nhãn thật, chỉ dùng để đánh giá sau cùng.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

np.random.seed(42)

## 2. Load Iris dataset

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature names:", iris.feature_names)
print("Species names:", iris.target_names)

Iris có:
- 150 observations
- 4 numerical features
- 3 species

Nhưng clustering chỉ dùng `X`.

## 3. Standardize predictors

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Mean:", X_scaled.mean(axis=0))
print("Std :", X_scaled.std(axis=0))

KMeans dùng khoảng cách Euclidean, nên standardization giúp các feature có scale tương đương:

\[
z = \frac{x-\mu}{\sigma}.
\]

## 4. Fit KMeans without labels

In [ ]:
k = 3

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20
)

clusters = kmeans.fit_predict(X_scaled)

print("First 20 cluster assignments:")
print(clusters[:20])

Điểm quan trọng: trong `fit_predict(X_scaled)` không có `y`.

Vì vậy đây là **unsupervised learning** đúng nghĩa.

## 5. Internal criterion 1 — Inertia

In [ ]:
inertia = kmeans.inertia_
print("Inertia for k=3:", inertia)

Inertia:

\[
\text{Inertia}
=
\sum_{i=1}^{n}
\|x_i-\mu_{c_i}\|^2
\]

- nhỏ hơn thường tốt hơn;
- nhưng inertia luôn giảm khi tăng số cluster K;
- vì vậy không nên chỉ nhìn inertia.

## 6. Internal criterion 2 — Silhouette score

In [ ]:
sil = silhouette_score(X_scaled, clusters)
print("Silhouette score for k=3:", sil)

Với mỗi điểm:

- \(a(i)\): khoảng cách trung bình tới các điểm cùng cluster.
- \(b(i)\): khoảng cách trung bình nhỏ nhất tới một cluster khác.

\[
s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}
\]

Khoảng giá trị:

\[
-1 \le s(i) \le 1
\]

- gần 1: cluster tốt;
- gần 0: nằm gần biên;
- âm: có thể bị xếp cluster chưa phù hợp.

## 7. Compare several values of K

In [ ]:
results = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)

    results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results_df["k"], results_df["inertia"], marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Inertia")
plt.title("Inertia vs K")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results_df["k"], results_df["silhouette"], marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Silhouette score")
plt.title("Silhouette score vs K")
plt.grid(alpha=0.3)
plt.show()

Cách đọc:

- **Inertia**: càng thấp càng tốt, nhưng luôn giảm khi K tăng.
- **Silhouette**: càng cao càng tốt.

Trong thực tế, ta kết hợp nhiều tiêu chí thay vì chỉ dựa vào một con số.

## 8. Stability — clusters có bền vững không?

Ta perturb dữ liệu bằng một lượng noise nhỏ, clustering lại nhiều lần, rồi so sánh với clustering ban đầu.

Dùng **Adjusted Rand Index (ARI)** để so sánh hai cách chia cluster.

Ở bước này ARI **không dùng species labels**.

In [ ]:
baseline_model = KMeans(n_clusters=3, random_state=42, n_init=20)
baseline_clusters = baseline_model.fit_predict(X_scaled)

stability_scores = []

for seed in range(30):
    rng = np.random.default_rng(seed)

    X_perturbed = X_scaled + rng.normal(
        loc=0,
        scale=0.05,
        size=X_scaled.shape
    )

    model = KMeans(n_clusters=3, random_state=seed, n_init=20)
    perturbed_clusters = model.fit_predict(X_perturbed)

    score = adjusted_rand_score(
        baseline_clusters,
        perturbed_clusters
    )

    stability_scores.append(score)

print("Mean stability ARI:", np.mean(stability_scores))
print("Min stability ARI :", np.min(stability_scores))
print("Max stability ARI :", np.max(stability_scores))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(stability_scores) + 1), stability_scores, marker="o")
plt.axhline(np.mean(stability_scores), linestyle="--", label="Mean stability")
plt.xlabel("Perturbation run")
plt.ylabel("ARI vs baseline clustering")
plt.title("Clustering stability under perturbation")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Nếu stability ARI cao, clustering ít thay đổi khi dữ liệu bị perturb nhẹ.

Đó chính là ý:

> **Do clusters persist under resampling or perturbation?**

## 9. External evaluation — bây giờ mới dùng species labels

In [ ]:
ari_external = adjusted_rand_score(y, clusters)
nmi_external = normalized_mutual_info_score(y, clusters)

print("External ARI:", ari_external)
print("External NMI:", nmi_external)

Bây giờ ta mới dùng `y`.

Mục tiêu là kiểm tra:

> Các cluster mà KMeans tự tìm ra có tương ứng với các species thật không?

ARI và NMI phù hợp vì cluster labels là arbitrary: cluster 0 hôm nay có thể trở thành cluster 2 ở một lần chạy khác.

## 10. Compare clusters with true species

In [ ]:
comparison = pd.crosstab(
    pd.Series(y, name="True species"),
    pd.Series(clusters, name="Cluster")
)

comparison.index = iris.target_names
comparison

Bảng trên giúp thấy:
- species nào được tách rõ;
- species nào bị trộn;
- cluster nào gần tương ứng với species nào.

## 11. Visualize clustering

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    X[:, 2],
    X[:, 3],
    c=clusters,
    s=45
)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("KMeans clustering on Iris")
plt.show()

## 12. Kết luận / câu trả lời kiểu thi

**In supervised learning**, predictions can be evaluated using held-out true labels.

**In clustering**, labels may be unavailable, so evaluation can rely on:

1. **Internal criteria**
   - inertia;
   - silhouette score.

2. **Stability**
   - check whether similar clusters persist after perturbation or resampling.

3. **External labels**
   - if labels are available only for evaluation, clustering should be performed without using them;
   - then cluster assignments can be compared with the labels afterward.

For the **Iris** dataset, KMeans uses only the numerical predictors. The species labels are used only after clustering to assess whether the discovered groups correspond to the true species.